In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import classification_report, confusion_matrix
import shap



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\ShehabYousef\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\ShehabYousef\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\ShehabYousef\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\ShehabYo

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
df = pd.read_csv('../Data/final_dataset.csv')

In [ ]:
y = df['target']
x = df.drop('target', axis=1) 

In [ ]:
train = df[df['datetime'] < '2015-06-01']
test  = df[df['datetime'] >= '2015-06-01']

drop_cols = ['target', 'failure_flag', 'datetime', 'last_maint_datetime']

X_train = train.drop(columns=drop_cols)
y_train = train['target']

X_test = test.drop(columns=drop_cols)
y_test = test['target']

In [ ]:
cat_features = ['comp', 'model']
num_features = [col for col in X_train.columns if col not in cat_features]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', 'passthrough', num_features)
    ]
)

In [ ]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 47.06781013163143


In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', xgb)
    
])
model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['comp', 'model']),
                                                 ('num', 'passthrough',
                                                  ['machineID', 'volt',
                                                   'rotate', 'pressure',
                                                   'vibration', 'error_count',
                                                   'maint_flag',
                                                   'days_since_maint', 'age',
                                                   'hour', 'dayofweek', 'month',
                                                   'is_weekend', 'volt_lag1',
                                                   'volt_lag3', 'rotate_lag1',
                                                   'rotate_lag3',
                                                   'pressu...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
threshold = 0.9

y_pred_custom = (y_prob >= threshold).astype(int)
print(classification_report(y_test, y_pred_custom))
print(confusion_matrix(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    504718
           1       0.80      0.86      0.83      9606

    accuracy                           0.99    514324
   macro avg       0.90      0.93      0.91    514324
weighted avg       0.99      0.99      0.99    514324

[[502617   2101]
 [  1342   8264]]
